### Abordagens off-line e on-line de data augmentation para treinamento de modelos de Deep Learning

Este notebook demonstra técnicas de data augmentation para o treinamento de um modelo de Deep Learning para detecção e classificação de tipos de gás.

Foi treinado um modelo de Multilayer Perceptron (MLP) para classificar as amostras do dataset MultimodalGasData. Este dataset possui dados coletados por sensores de detecção de gás de óxido metálico. As amostras foram classificadas em 4 classes: sem gás, fumaça, perfume (spray), e, mix de fumaça e perfume.

#### Importar bibliotecas

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import tensorflow as tf
import time
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

#### Carregar o dataset

In [2]:
df = pd.read_csv('../dataset/Gas_Sensors_Measurements.csv', sep = ',')

#### Verificar o shape do dataset

In [3]:
df.shape

(6400, 10)

#### Verificar o tipo das features

In [7]:
df = df.drop(columns = ['Serial Number', 'Corresponding Image Name'])
df.dtypes

MQ2       int64
MQ3       int64
MQ5       int64
MQ6       int64
MQ7       int64
MQ8       int64
MQ135     int64
Gas      object
dtype: object

#### Verificar algumas amostras de exemplo

In [8]:
pd.set_option('display.max_columns', None)
df.head()

,MQ2,MQ3,MQ5,MQ6,MQ7,MQ8,MQ135,Gas
0,555,515,377,338,666,451,416,NoGas
1,555,516,377,339,666,451,416,NoGas
2,556,517,376,337,666,451,416,NoGas
3,556,516,376,336,665,451,416,NoGas
4,556,516,376,337,665,451,416,NoGas


#### Verificar se há dados ausentes

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6400 entries, 0 to 6399
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   MQ2     6400 non-null   int64 
 1   MQ3     6400 non-null   int64 
 2   MQ5     6400 non-null   int64 
 3   MQ6     6400 non-null   int64 
 4   MQ7     6400 non-null   int64 
 5   MQ8     6400 non-null   int64 
 6   MQ135   6400 non-null   int64 
 7   Gas     6400 non-null   object
dtypes: int64(7), object(1)
memory usage: 400.1+ KB


#### Verificar a distribuição das amostras pelas classes

In [10]:
instances_count = df['Gas'].value_counts().sum()
class_distribution = df['Gas'].value_counts().to_dict()
for key, value in class_distribution.items() :
    print(f"Class = {key}   Qty = {value:6.0f}   Percentage = {(value / instances_count * 100):.2f} %")

Class = NoGas   Qty =   1600   Percentage = 25.00 %
Class = Perfume   Qty =   1600   Percentage = 25.00 %
Class = Smoke   Qty =   1600   Percentage = 25.00 %
Class = Mixture   Qty =   1600   Percentage = 25.00 %


#### Função para gerar data augmentation off-line e classe para gerar data augmetation on-line

In [54]:
# New CSV header
columns = ["MQ2"
    ,"MQ3"
    ,"MQ5"
    ,"MQ6"
    ,"MQ7"
    ,"MQ8"
    ,"MQ135"
]

target_to_num_dict = {'NoGas': 0, 'Smoke': 1, 'Perfume': 2, 'Mixture': 3}
target_to_str_dict = {0: 'NoGas', 1: 'Smoke', 2: 'Perfume', 3: 'Mixture'}

class OnlineDataGenerator(Sequence):
    def __init__(self, x_set, y_set, batch_size, augment = False):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(self.x.shape[0])
    def __len__(self):
        return int(np.ceil(len(self.x) / self.batch_size))
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = self.x[batch_indices].copy()
        batch_y = self.y[batch_indices]
        # Apply augmentation with 50% probability
        if self.augment and random.random() < 0.5:
            augmented_batch = []
            for sample, y in zip(batch_x, batch_y):
                # Simulate Gaussian Noise
                noise = np.random.normal(0, 0.03, sample.shape)
                aug_sample = sample + noise
                augmented_batch.append(aug_sample)
            batch_x = np.stack(augmented_batch)
        return batch_x, batch_y
    def on_epoch_end(self):
        np.random.shuffle(self.indices)

def generate_offline_augmentation(X_train, y_train, dataset_dir, replication_factor = 4):
    augmented_samples = []
    augmented_labels = []
    print("\nStarting offline data augmentation...")
    for idx in range(len(X_train)):
        sample = X_train[idx]
        label = y_train[idx]
        augmented_samples.append(sample)
        augmented_labels.append(label)
        for rep in range(replication_factor):
            # Simulate Gaussian Noise
            noise = np.random.normal(0, 0.03, sample.shape)
            aug_sample = sample + noise
            augmented_samples.append(aug_sample)
            augmented_labels.append(label)
    X_train_final = np.array(augmented_samples, dtype = "float64")
    y_train_final = np.array(augmented_labels)
    print("\nOffline augmentation finished.")
    print(f"\nOriginal dataset size: {len(X_train)}")
    print(f"Final dataset size: {len(X_train_final)}")
    unique_classes, counts = np.unique(y_train_final, return_counts = True)
    print("\nClass distribution:")
    for cls, count in zip(unique_classes, counts):
        print(f"Class {cls}: {count}")
    y_train_name = [target_to_str_dict[val] for val in y_train_final]
    new_df = pd.DataFrame(X_train_final, columns = columns)
    new_df["Gas"] = y_train_name
    # Save new CSV
    filename = "aug_Gas_Sensors_Measurements.csv"
    full_path = os.path.join(dataset_dir, filename)
    os.makedirs(dataset_dir, exist_ok = True)
    new_df.to_csv(full_path, index = True)
    print(f"\nCSV saved successfully: {filename}")
    return X_train_final, y_train_final

#### Função para criação do modelo de MLP

In [12]:
def build_model(input_shape = (None, 7)):
    inputs = layers.Input(shape = input_shape)
    x = layers.Dense(32, activation="relu")(inputs)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(4, activation = "softmax")(x)
    model = models.Model(inputs = inputs, outputs = outputs)
    model.compile(
        optimizer = Adam(learning_rate = 0.001),
        loss = "sparse_categorical_crossentropy",
        metrics = ["accuracy"]
    )
    return model

#### Função para treino do modelo

In [13]:
def train_model(model, X_train, y_train, X_val, y_val, epochs = 50, batch_size = 64, augmentation = False, online_aug = False):
    callbacks = [
        EarlyStopping(
            monitor = "val_loss",
            patience = 10,
            restore_best_weights = True,
            verbose = 1
        )
    ]
    history = None
    if online_aug == False:
        history = model.fit(
            X_train, y_train,
            validation_data = (X_val, y_val),
            epochs = epochs,
            batch_size = batch_size,
            callbacks = callbacks
        )
    else:
        train_gen = OnlineDataGenerator(X_train, y_train, batch_size = batch_size, augment = True)
        val_gen = OnlineDataGenerator(X_val, y_val, batch_size = batch_size, augment = False) # No augment.
        history = model.fit(train_gen,
            validation_data = val_gen,
            epochs = epochs,
            batch_size = batch_size,
            callbacks = callbacks
        )
    current_dir = os.getcwd()
    config_file_name = None
    if augmentation == False:
        config_file_name = 'modelconfig.keras'
    elif online_aug == False:
        config_file_name = 'modelconfig_offline.keras'
    else:
        config_file_name = 'modelconfig_online.keras'
    config_path = os.path.join(current_dir, config_file_name)
    model.save(config_path)
    return history

#### Função para avaliação da performance do modelo

In [14]:
def evaluate_model(model, X_test, y_test):
    loss, acc = model.evaluate(X_test, y_test, verbose = 0)
    print(f"\nTest Loss: {loss:.4f}")
    print(f"Test Accuracy: {acc:.4f}")
    y_pred = np.argmax(model.predict(X_test), axis = 1)
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_test, y_pred))

#### Função para dividir o dataset em treino, validação e teste

In [15]:
def split_dataset(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size = 0.1,
        random_state = 42,
        stratify = y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train,
        y_train,
        test_size = 0.1,
        random_state = 42,
        stratify = y_train
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

#### Executar o pipeline de treinamento completo sem data augmentation

In [16]:
DATASET_DIR = "../dataset"
y = df['Gas'].map(target_to_num_dict).to_numpy()
X = df.drop(columns = ['Gas']).to_numpy(dtype = np.float64)

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)
print("X_train.shape: ", X_train.shape)
print("X_val.shape: ", X_val.shape)
print("X_test.shape: ", X_test.shape)
model = build_model(input_shape = X_train.shape[1:])
model.summary()
start_time = time.time()
history = train_model(model, X_train, y_train, X_val, y_val, augmentation = False, online_aug = False)
evaluate_model(model, X_test, y_test)

elapsed_seconds = time.time() - start_time
print("\nTime taken for training: ", time.strftime("%H:%M:%S", time.gmtime(elapsed_seconds)))

X_train.shape:  (5184, 7)
X_val.shape:  (576, 7)
X_test.shape:  (640, 7)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 7)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,580 (17.89 KB)

 Trainable params: 4,580 (17.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4331 - loss: 8.9743 - val_accuracy: 0.6059 - val_loss: 1.4196
Epoch 2/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7861 - loss: 0.6522 - val_accuracy: 0.8160 - val_loss: 0.4860
Epoch 3/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8736 - loss: 0.3301 - val_accuracy: 0.9115 - val_loss: 0.2249
Epoch 4/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8935 - loss: 0.2658 - val_accuracy: 0.8733 - val_loss: 0.2595
Epoch 5/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8972 - loss: 0.2433 - val_accuracy: 0.8976 - val_loss: 0.2536
Epoch 6/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8688 - loss: 0.3531 - val_accuracy: 0.9392 - val_loss: 0.1685
Epoch 7/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8910 - loss: 0.2527 - val_accuracy: 0.9306 - val_loss: 0.1831
Epoch 8/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9122 - loss: 0.2077 - val_accuracy: 0.9184 - val_loss:

#### Executar o pipeline de treinamento completo com data augmentation off-line

In [55]:
DATASET_DIR = "../dataset"
y = df['Gas'].map(target_to_num_dict).to_numpy()
X = df.drop(columns = ['Gas']).to_numpy(dtype = np.float64)

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)
print("X_train.shape: ", X_train.shape)
print("X_val.shape: ", X_val.shape)
print("X_test.shape: ", X_test.shape)
X_train, y_train = generate_offline_augmentation(X_train, y_train, DATASET_DIR)
model = build_model(input_shape = X_train.shape[1:])
model.summary()
start_time = time.time()
history = train_model(model, X_train, y_train, X_val, y_val, augmentation = True, online_aug = False)
evaluate_model(model, X_test, y_test)

elapsed_seconds = time.time() - start_time
print("\nTime taken for training: ", time.strftime("%H:%M:%S", time.gmtime(elapsed_seconds)))

X_train.shape:  (5184, 7)
X_val.shape:  (576, 7)
X_test.shape:  (640, 7)

Starting offline data augmentation...

Offline augmentation finished.

Original dataset size: 5184
Final dataset size: 25920

Class distribution:
Class 0: 6480
Class 1: 6480
Class 2: 6480
Class 3: 6480

CSV saved successfully: aug_Gas_Sensors_Measurements.csv


Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_19 (InputLayer)          │ (None, 7)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_76 (Dense)                     │ (None, 32)                  │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_77 (Dense)                     │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_78 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_79 (Dense)                     │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,580 (17.89 KB)

 Trainable params: 4,580 (17.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7297 - loss: 2.1371 - val_accuracy: 0.7222 - val_loss: 1.2478
Epoch 2/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8275 - loss: 0.4000 - val_accuracy: 0.7465 - val_loss: 0.6946
Epoch 3/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8563 - loss: 0.3563 - val_accuracy: 0.9045 - val_loss: 0.2434
Epoch 4/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8706 - loss: 0.3154 - val_accuracy: 0.8837 - val_loss: 0.2725
Epoch 5/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8851 - loss: 0.2714 - val_accuracy: 0.9062 - val_loss: 0.2226
Epoch 6/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8836 - loss: 0.2907 - val_accuracy: 0.9045 - val_loss: 0.2161
Epoch 7/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8877 - loss: 0.2921 - val_accuracy: 0.9444 - val_loss: 0.1747
Epoch 8/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9042 - loss: 0.2292 - val_accuracy: 0.

#### Executar o pipeline de treinamento completo com data augmentation on-line

In [47]:
DATASET_DIR = "../dataset"
y = df['Gas'].map(target_to_num_dict).to_numpy()
X = df.drop(columns = ['Gas']).to_numpy(dtype = np.float64)

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)
print("X_train.shape: ", X_train.shape)
print("X_val.shape: ", X_val.shape)
print("X_test.shape: ", X_test.shape)
model = build_model(input_shape = X_train.shape[1:])
model.summary()
start_time = time.time()
history = train_model(model, X_train, y_train, X_val, y_val, augmentation = True, online_aug = True)
evaluate_model(model, X_test, y_test)

elapsed_seconds = time.time() - start_time
print("\nTime taken for training: ", time.strftime("%H:%M:%S", time.gmtime(elapsed_seconds)))

X_train.shape:  (5184, 7)
X_val.shape:  (576, 7)
X_test.shape:  (640, 7)


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)          │ (None, 7)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_64 (Dense)                     │ (None, 32)                  │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_65 (Dense)                     │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_66 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_67 (Dense)                     │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,580 (17.89 KB)

 Trainable params: 4,580 (17.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


C:\Program Files\Python311\keras3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5883 - loss: 11.4789 - val_accuracy: 0.8038 - val_loss: 0.4212
Epoch 2/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8677 - loss: 0.2987 - val_accuracy: 0.9097 - val_loss: 0.2366
Epoch 3/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8939 - loss: 0.2402 - val_accuracy: 0.8889 - val_loss: 0.2397
Epoch 4/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8972 - loss: 0.2403 - val_accuracy: 0.9410 - val_loss: 0.1936
Epoch 5/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8945 - loss: 0.2364 - val_accuracy: 0.8993 - val_loss: 0.2163
Epoch 6/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8787 - loss: 0.2638 - val_accuracy: 0.9236 - val_loss: 0.1901
Epoch 7/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9064 - loss: 0.2124 - val_accuracy: 0.8872 - val_loss: 0.2178
Epoch 8/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9057 - loss: 0.2105 - val_accuracy: 0.9340 - val_loss: 0.1505
Ep